# Intento de forma no lineal

In [1]:
import numpy as np
import ufl
from dolfinx import default_scalar_type, fem, mesh
from mpi4py import MPI
from petsc4py import PETSc

comm = MPI.COMM_WORLD
cell_type = mesh.CellType.tetrahedron

# Params
L = 4
H = 1
W = 1
nx = 20
ny = 10
nz = 10
tdim = 3

domain = mesh.create_box(
    comm,
    [[0.0, 0.0, 0.0], [L, H, W]],
    [nx, ny, nz],
    cell_type=cell_type
)

V_space = fem.functionspace(domain, ("Lagrange", 1, (tdim,)))
u = fem.Function(V_space)
du = ufl.TrialFunction(V_space)
v = ufl.TestFunction(V_space)


$$
\nabla_0 \cdot P + B = 0 \\
\int_{\Omega_0} P(u) : \nabla_0 v dV = \int_{\Omega_0} B \cdot v dV + \int_{\partial \Omega_0} T_0 \cdot v d S \\
R(u; v) = \int_{\Omega_0} P(u) : \nabla_0 v dV - \int_{\Omega_0} B \cdot v dV - \int_{\partial \Omega_0} T_0 \cdot v d S 
$$

In [2]:
lam = fem.Constant(domain, default_scalar_type(1e10))
mu = fem.Constant(domain, default_scalar_type(1e10))
B = fem.Constant(domain, [0,0, -9.81*7850])

def saint_venant_kirchhoff(u, lam, mu, tdim=3):
    I = ufl.Identity(tdim)
    F = I + ufl.grad(u)
    E = 1/2 * (ufl.dot(F.T,  F) - I)
    S = lam * ufl.tr(E) * I + 2 * mu * E
    P = ufl.dot(F, S)
    return P


In [3]:
def P(v): return saint_venant_kirchhoff(v, lam, mu, tdim=tdim) 

#B_final = np.array([0.0, 0.0, -9.81], dtype=default_scalar_type)
#B = fem.Constant(domain, default_scalar_type * B_final)

def residual(u, v, P, B):
    return ufl.inner(P(u), ufl.grad(v)) * ufl.dx - ufl.dot(B, v) * ufl.dx

R = residual(u, v, P, B)
Dr = ufl.derivative(R, u, du)

def left_boundary(x):
    return np.isclose(x[0], 0.0)

left_dofs = fem.locate_dofs_geometrical(V_space, left_boundary)
bc = fem.dirichletbc(np.zeros(tdim, dtype=default_scalar_type), left_dofs, V_space)

In [4]:
from dolfinx.fem.petsc import NonlinearProblem
from dolfinx.nls.petsc import NewtonSolver

problem = NonlinearProblem(
    R, u, 
    bcs=[bc], J=Dr,
)

solver = NewtonSolver(comm, problem)

solver.rtol = 1e-4
solver.atol = 1e-5
solver.max_it = 200
solver.report = True

ksp = solver.krylov_solver
ksp.setType("preonly")

pc = ksp.getPC()
pc.setType("lu")

# para que sea mas rapido freo
pc.setFactorSolverType("mumps")


In [5]:
# Se resuelve por pasos de fuerza

nsteps = 100
for step in range(1, nsteps+1):
    factor = step / nsteps
    B.value[:] = [0, 0, -9.81*7850 * factor]
    

    nit, flag = solver.solve(u)
    print(f"Step {step}/{nsteps}: Number of Newton iterations: {nit}, Converged: {flag}")
    u.x.scatter_forward()


Step 1/100: Number of Newton iterations: 2, Converged: True
Step 2/100: Number of Newton iterations: 2, Converged: True
Step 3/100: Number of Newton iterations: 2, Converged: True
Step 4/100: Number of Newton iterations: 2, Converged: True
Step 5/100: Number of Newton iterations: 2, Converged: True
Step 6/100: Number of Newton iterations: 2, Converged: True
Step 7/100: Number of Newton iterations: 2, Converged: True
Step 8/100: Number of Newton iterations: 2, Converged: True
Step 9/100: Number of Newton iterations: 2, Converged: True
Step 10/100: Number of Newton iterations: 2, Converged: True
Step 11/100: Number of Newton iterations: 2, Converged: True
Step 12/100: Number of Newton iterations: 2, Converged: True
Step 13/100: Number of Newton iterations: 2, Converged: True
Step 14/100: Number of Newton iterations: 2, Converged: True
Step 15/100: Number of Newton iterations: 2, Converged: True
Step 16/100: Number of Newton iterations: 2, Converged: True
Step 17/100: Number of Newton ite

In [9]:
import pyvista as pv
from dolfinx import plot


# 1. Malla no deformada
topology, cell_types, geometry = plot.vtk_mesh(domain, domain.topology.dim)
grid_undeformed = pv.UnstructuredGrid(topology, cell_types, geometry)

# 2. Malla deformada (evaluando el campo de desplazamientos u_h)
V_plot, _ = u.function_space.sub(0).collapse() if u.function_space.num_sub_spaces > 0 else (u.function_space, None)
plot_topology, plot_cell_types, plot_geometry = plot.vtk_mesh(V_space)
grid_deformed = pv.UnstructuredGrid(plot_topology, plot_cell_types, plot_geometry)

# Agregar los desplazamientos (completando a 3D para PyVista si es 2D)
u_values = u.x.array.reshape((plot_geometry.shape[0], V_space.dofmap.index_map_bs))
displacement_3d = np.zeros((plot_geometry.shape[0], 3))
displacement_3d[:, :domain.geometry.dim] = u_values

grid_deformed.point_data["Desplazamiento"] = displacement_3d

# Factor de amplificación para visualizar mejor la deformación
factor = 1e3
warped = grid_deformed.warp_by_vector("Desplazamiento", factor=factor)

# 3. Graficar ambas configuraciones superpuestas
plotter = pv.Plotter()
plotter.add_text(f"Configuración no deformada vs deformada (Escala x{factor:,.0e})", font_size=11)
plotter.add_mesh(grid_undeformed, style="wireframe", color="gray", line_width=1.5, label="No deformada")
plotter.add_mesh(warped, scalars="Desplazamiento", show_edges=True, cmap="viridis", label="Deformada")
plotter.add_legend()
plotter.view_xy()
plotter.show()

Widget(value='<iframe src="http://localhost:41055/index.html?ui=P_0x7faf5385df10_3&reconnect=auto" class="pyvi…